In [1]:
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [2]:
financial_corpus = [
   "A stock market is a public market for the trading of company stock and derivatives at an agreed price. It is a key component of the financial system.",
"Inflation refers to the rate at which the general level of prices for goods and services is rising, and subsequently, purchasing power is falling.",
"A bond is a fixed-income instrument that represents a loan made by an investor to a borrower, typically corporate or governmental.",
"The gross domestic product (GDP) is the total value of everything produced by all the people and companies in the country.",
"Interest rates are the cost of borrowing money or the return for investing money. Central banks influence interest rates to control inflation and stabilize the currency.",
"An exchange-traded fund (ETF) is a type of investment fund and exchange-traded product, meaning they are traded on stock exchanges.",
"A mutual fund is a professionally managed investment fund that pools money from many investors to purchase securities.",
"Risk management in finance involves identifying, assessing, and prioritizing risks followed by coordinated efforts to minimize, monitor, and control the probability or impact of",
"Corporate finance deals with the capital structure of corporations, including the funding and the actions that management takes to increase the value of the company.",
"Quantitative easing is a monetary policy whereby a central bank buys government securities or other securities from the market to lower interest rates and increase the money supply."
]

In [3]:
from sentence_transformers import SentenceTransformer, util

#Load a pretrainer retriever model
retriever = SentenceTransformer('msmarco-distilbert-base-v4')

#Encode the domain corpus
corpus_embeddings = retriever.encode(financial_corpus, convert_to_tensor=True)

In [4]:
from transformers import BartForConditionalGeneration, BartTokenizer

#Load a pre-trained XLNet Model and tokenizer
bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
bart_tokenizer.pad_token = bart_tokenizer.eos_token

bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large')
bart_model.generation_config.pad_token_id = bart_tokenizer.eos_token_id

In [5]:
def retrieve_and_generate(query, top_k = 5):
    #Encode the query
    query_embedding = retriever.encode(query, convert_to_tensor=True)

    #Retrieve top_k relevant documents
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k = top_k)[0]
    retrieve_docs = [financial_corpus[hit['corpus_id']] for hit in hits]

    #Concatenate retrieved documents as context
    context = " ".join(retrieve_docs)

    #Encode context and query
    input_ids = bart_tokenizer.encode(context + query, return_tensors='pt')

    #Generate response
    output = bart_model.generate(input_ids, max_length = 300, num_return_sequences=1)
    response = bart_tokenizer.decode(output[0], skip_special_tokens=True)

    return response

#Sample query
query = "What is stock market?"
response = retrieve_and_generate(query)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


A stock market is a public market for the trading of company stock and derivatives at an agreed price. It is a key component of the financial system. Inflation refers to the rate at which the general level of prices for goods and services is rising, and subsequently, purchasing power is falling. Quantitative easing is a monetary policy whereby a central bank buys government securities or other securities from the market to lower interest rates and increase the money supply. A mutual fund is a professionally managed investment fund that pools money from many investors to purchase securities. An exchange-traded fund (ETF) is a type of investment fund which pools money with other investors to buy and sell securities on the stock exchanges.What is stock market?


In [6]:
#Install the datasets library
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable


In [7]:
#install the faiss-cpu library
!pip install faiss-cpu
#required for RAG retriever

Defaulting to user installation because normal site-packages is not writeable


In [8]:
#install the transformers library
!pip install transformers[torch]

Defaulting to user installation because normal site-packages is not writeable


In [9]:
#import the required libraries
from transformers import BertTokenizer, BertForQuestionAnswering, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch


In [10]:
#Create a sample task-specific corpus for questions and answers
qa_corpus = [
{
    "context": "Diabetes is a chronic condition characterized by high levels of sugar (glucose) in the blood. Common symptoms include increased thirst, frequent urination, and uneasiness.",
"question": "What are the symptoms of diabetes?",
"answer": "Common symptoms include increased thirst, frequent urination, and unexplained weight loss."
},
{
"context": "Hypertension, also known as high blood pressure, is a condition in which the force of the blood against the artery walls is too high. Often hypertension has no symptoms.",
"question": "What is hypertension?",
"answer": "Hypertension is a condition in which the force of the blood against the artery walls is too high."
},
{
"context": "Asthma is a common long-term inflammatory disease of the airways of the lungs. Symptoms include episodes of wheezing, coughing, chest tightness, and shortness of breath.",
"question": "What are the symptoms of asthma?",
"answer": "Symptoms include episodes of wheezing, coughing, chest tightness, and shortness of breath."
},
{
"context": "Coronary artery disease (CAD) is the most common type of heart disease. It occurs when the coronary arteries that supply blood to the heart muscle become hardened and narrowed due to the buildup of cholesterol and cough.",
"question": "What is coronary artery disease?",
"answer": "Coronary artery disease occurs when the coronary arteries that supply blood to the heart muscle become hardened and narrowed due to the buildup of cholesterol and cough."
},
{    
"context": "Osteoarthritis is the most common form of arthritis, affecting millions of people worldwide. It occurs when the protective cartilage that cushions the ends of the flexibility.",
"question": "What are the symptoms of osteoarthritis?",
"answer": "Symptoms include pain, stiffness, and loss of flexibility."
},
{
"context": "Influenza, commonly known as the flu, is an infectious disease caused by an influenza virus. Symptoms can be mild to severe and commonly include a high fever, running nose.",
"question": "What are the symptoms of influenza?",
"answer": "Symptoms commonly include a high fever, runny nose, sore throat, muscle and joint pain, headache, coughing, and feeling tired."
},
{
"context": "Anemia is a condition in which you lack enough healthy red blood cells to carry adequate oxygen to your body's tissues. Having anemia can make you feel tired and  weakness, dizziness, shortness of breath, and pale skin.",
"question": "What are the symptoms of anemia?",
"answer": "Symptoms can include tiredness, weakness, dizziness, shortness of breath, and pale skin."
},
{
"context": "Chronic obstructive pulmonary disease (COPD) is a chronic inflammatory lung disease that causes obstructed airflow from the lungs. Symptoms include breathing difficulties.",
"question": "What are the symptoms of COPD?",
"answer": "Symptoms include breathing difficulty, cough, mucus (sputum) production, and wheezing."
},
{
"context": "Gastroesophageal reflux disease (GERD) is a chronic digestive disease. GERD occurs when stomach acid or, occasionally, stomach content, flows back into your food pipe (esophagus). The backwash (acid reflux) irritation.",
"question": "What is GERD?",
"answer": "GERD is a chronic digestive disease that occurs when stomach acid or stomach content flows back into your food pipe (esophagus). The backwash (acid reflux) irritation."
},
{
"context": "Chronic kidney disease (CKD) means your kidneys are damaged and can't filter blood the way they should. The disease is called 'chronic' because the damage to your legs, ankles, or feet, and shortness of breath.",
"question": "What are the symptoms of chronic kidney disease?",
"answer": "Symptoms may include fatigue, swelling in your legs, ankles, or feet, and shortness of breath."
}
]

In [11]:
#Write a QA dataset function for questions and answers
class QADataset(Dataset):
    def __init__(self, data, tokenizer, max_length = 512):
        self.examples = []
        for item in data:
            inputs = tokenizer. encode_plus(
                item['question'], item['context'],
                add_special_tokens=True,
                max_length = max_length,
                padding = 'max_length',
                truncation=True,
                return_tensors = 'pt'
        )
        start_positions = inputs.input_ids.squeeze().tolist().index(tokenizer.encode(item['answer'], add_special_tokens=False)[0])
        end_positions = start_positions + len(tokenizer.encode(item['answer'], add_special_tokens=False)) - 1
        self.examples.append({
            'input_ids': inputs.input_ids.squeeze(),
            'attention_mask': inputs.attention_mask.squeeze(),
            'start_positions': torch.tensor(start_positions),
            'end_positions': torch.tensor(end_positions)
        })
    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]
            

In [12]:
#Load a pre-trainer BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForQuestionAnswering.from_pretrained('bert-base-uncased')

#Prepare datasets 
train_dataset= QADataset(qa_corpus, tokenizer)


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
!pip install transformers[torch]

Defaulting to user installation because normal site-packages is not writeable


In [15]:
import sys
!"{sys.executable}" -m pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/619.4 MB ? eta -:--:--
   ---------------------------------------- 1.3/619.4 MB 8.4 MB/s eta 0:01:14
   ---------------------------------------- 3.7/619.4 MB 10.4 MB/s eta 0:01:00
   ---------------------------------------- 6.3/619.4 MB 11.4 MB/s eta 0:00:54
    --------------------------------------- 8.9/619.4 MB 11.5 MB/s eta 0:00:53
    --------------------------------------- 11.3/619.4 MB 11.8 MB/s eta 0:00:52
    --------------------------------------- 13.9/619.4 MB 11.8 MB/s eta 0:00:52
   - -------------------------------------- 16.3/619.4 MB 11.8 MB/s eta 0:00:52
   - -------------------------------------- 19.1/619.4 MB 12.0 MB/s eta 0:00:51
   - -------------------------------------- 21.5/619.4 MB 11.9 MB/s eta 0:00:51
   - -------------------------------------- 24.1/619.4 MB 12.0 MB/s eta 0:00:50

  You can safely remove it manually.


In [16]:
import torch, accelerate, transformers
print("torch:", torch.__version__, "cuda?", torch.cuda.is_available())
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)


torch: 2.8.0+cpu cuda? False
accelerate: 1.10.1
transformers: 4.56.1


In [17]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "accelerate", "config", "default"])

CompletedProcess(args=['C:\\Program Files\\Python311\\python.exe', '-m', 'accelerate', 'config', 'default'], returncode=1)

In [18]:
from accelerate.utils import write_basic_config
write_basic_config()

Configuration already exists at C:\Users\sampa/.cache\huggingface\accelerate\default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


False

In [19]:
import sys, importlib, transformers, accelerate
from transformers.utils import is_accelerate_available, ACCELERATE_MIN_VERSION
print("py:", sys.executable)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__, "min required:", ACCELERATE_MIN_VERSION)
print("find_spec(accelerate):", importlib.util.find_spec("accelerate") is not None)
print("is_accelerate_available():", is_accelerate_available())


py: C:\Program Files\Python311\python.exe
transformers: 4.56.1
accelerate: 1.10.1 min required: 0.26.0
find_spec(accelerate): True
is_accelerate_available(): True


In [ ]:
import sys
!"{sys.executable}" -m pip install -U "accelerate>=0.26.0,<2.0.0" "transformers>=4.56.0,<4.57.0" packaging


In [ ]:
from accelerate.utils import write_basic_config
write_basic_config()


In [21]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,              # start small
    per_device_train_batch_size=4,
    save_strategy="no",              # simplest path
    logging_steps=50,
    report_to="none",
    eval_strategy="no",              # correct name for 4.56
    no_cuda=True,                    # CPU-only build
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)
trainer.train()


Step,Training Loss


TrainOutput(global_step=3, training_loss=4.447808583577474, metrics={'train_runtime': 11.5845, 'train_samples_per_second': 0.259, 'train_steps_per_second': 0.259, 'total_flos': 783890270208.0, 'train_loss': 4.447808583577474, 'epoch': 3.0})

In [22]:
def generate_response_qa(question, context, model, tokenizer):
    inputs = tokenizer. encode_plus(question,context,return_tensors='pt')
    input_ids = inputs['input_ids'].tolist()[0]
    text_tokens = tokenizer.convert_ids_to_tokens(input_ids)

    outputs = model(**inputs)
    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1

    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end]))
    return answer

#Sample question and context
context = "Diabetes is a chronic condition characterized by high levels of sugar (glucose) in the blood. Common symptoms include increased thirst, frequent urination, and unexplained symptoms."
question = "What are the symptoms of diabetes?"

response = generate_response_qa(question, context, model, tokenizer)
print(response)
    

symptoms of diabetes ? [SEP] diabetes is a chronic condition characterized by high levels of
